# Model 0 real run: three-seed test evaluation (Part E) and cold-start (Part F), Sneha

Before running: **Runtime → Change runtime type → GPU** (T4 or better).

Uses Abhinita's three Model 0 runs (grid index 7: hidden size 128, dropout 0.5, learning rate 0.0005, seeds 42, 43, 44).
Her original Drive folders are not touched. Copies and every output go to `MyDrive/DATA 298A/model0_real_results`, so nothing is lost if Colab disconnects.

**Run each cell once.** The test split is scored once per seed; a second run reuses the saved result instead of re-scoring.

## 0. One-time setup in Google Drive (before opening Colab)
Upload these to `MyDrive/DATA 298A` (drag them from Finder into the Drive window):
1. `model0_real_run_files.zip` (the team repo main, `a13ad30`).
2. The folder `model0_seed_runs` from `~/Desktop/DATA 298A`. It holds `seed_42`, `seed_43` and `seed_44`, each with `best.pt`, `run_config.json`, `history.csv` and `summary.json`, taken from Abhinita's `part_d_grid_results/grid_full_20261107T200443Z_69f3bc0e` (`run_07_seed_42`, `winner_repeats/seed_43`, `winner_repeats/seed_44`).

`part_e_f_data.zip` is already in `MyDrive/DATA 298A` from the first run.

## 1. Mount Drive and set paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os

DRIVE = '/content/drive/MyDrive/DATA 298A'
ZIP = f'{DRIVE}/model0_real_run_files.zip'
DATA_ZIP = f'{DRIVE}/part_e_f_data.zip'      # the 14 frozen data files
SEED_LINKS = f'{DRIVE}/model0_seed_runs'      # seed_42, seed_43, seed_44 from Abhinita's runs
OUT = f'{DRIVE}/model0_real_results'          # all copies and outputs
REPO = '/content/data298a-team5-topic22'
GRID_INDEX = 7

for path in (ZIP, DATA_ZIP, SEED_LINKS):
    print('found' if os.path.exists(path) else 'MISSING', path)
assert os.path.exists(ZIP) and os.path.exists(DATA_ZIP) and os.path.exists(SEED_LINKS), 'Do step 0 first'
os.makedirs(OUT, exist_ok=True)

## 2. Unpack the code (team repo main) and the frozen data

In [ ]:
!rm -rf {REPO} && mkdir -p {REPO}
!unzip -q -o "{ZIP}" -d {REPO}
%cd {REPO}
!mkdir -p data && unzip -q -o "{DATA_ZIP}" -d data
!find data/processed -type f | sort

## 3. Install the pinned data libraries
Same as Abhinita's notebook 10: NumPy, pandas and PyArrow are pinned; Colab's own PyTorch is kept, as in the training runs.

In [ ]:
!pip install -q numpy==2.2.6 pandas==2.3.2 pyarrow==21.0.0
# Checked in a fresh process, which is how every step below runs.
!python -c "import platform, torch, numpy, pandas, pyarrow; print('python', platform.python_version(), '| torch', torch.__version__, '| numpy', numpy.__version__, '| pandas', pandas.__version__, '| pyarrow', pyarrow.__version__); print('GPU:', torch.cuda.is_available())"
import torch
assert torch.cuda.is_available(), 'Switch the runtime to GPU'

## 4. Run all tests
Everything should be `ok`. `MultiSeedTests` must say ok, **not skipped**.

In [ ]:
!python -B -m unittest discover -s tests -v 2>&1 | tail -25

## 5. Copy the three runs into my results folder and check them
Each `best.pt` is checked against the hash in Abhinita's `handoff_verification.json`.

In [ ]:
import glob, hashlib, json, shutil

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()

handoff = json.load(open('docs/project_notes/model0_full_grid_20261107/handoff_verification.json'))
expected = {r['seed']: r['checkpoint_sha256'] for r in handoff['runs']}
RUNS = {}
for folder in sorted(glob.glob(f'{SEED_LINKS}/*')):
    config_path = os.path.join(folder, 'run_config.json')
    if not os.path.exists(config_path):
        print('skipping (no run_config.json):', folder)
        continue
    seed = json.load(open(config_path))['training']['seed']
    target = f'{OUT}/runs/seed_{seed}'
    if not os.path.exists(f'{target}/best.pt'):
        os.makedirs(target, exist_ok=True)
        for name in ('best.pt', 'run_config.json', 'history.csv', 'summary.json'):
            shutil.copy2(os.path.join(folder, name), target)
    assert sha256(f'{target}/best.pt') == expected[seed], f'seed {seed}: best.pt hash differs from the handoff'
    RUNS[seed] = target
    print(f'seed {seed}: ok ->', target)
assert sorted(RUNS) == [42, 43, 44], f'Expected seeds 42, 43, 44, found {sorted(RUNS)}'

## 6. Part E: three-seed test evaluation
Reloads each checkpoint, checks it reproduces its saved validation NDCG@10, then scores the test split once.
Writes `test_evaluation.json` and `test_ranks.npz` into each run copy, and the tables into `model0_test_3seeds`.

In [ ]:
TEST_OUT = f'{OUT}/model0_test_3seeds'
if os.path.exists(f'{TEST_OUT}/aggregate.json'):
    print('Already done:', TEST_OUT)
else:
    !python -B -m src.evaluation.multi_seed evaluate --device cuda \
      --config configs/config.json --grid-index {GRID_INDEX} \
      --run-dirs "{RUNS[42]}" "{RUNS[43]}" "{RUNS[44]}" \
      --output-dir "{TEST_OUT}"

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(f'{TEST_OUT}/evaluation_table.md').read()))

## 7. Part F: cold-start reference (no model)
Expected: popularity on new products NDCG@10 0.0000, warm control 0.0134.

In [ ]:
COLD_REF = f'{OUT}/cold_reference'
if os.path.exists(COLD_REF):
    print('Already done:', COLD_REF)
else:
    !python -B -m src.evaluation.cold_start reference --output-dir "{COLD_REF}"
display(Markdown(open(f'{COLD_REF}/cold_start_table.md').read()))

## 8. Part F: train Model 0 on the cold-start data, one copy per seed
Same settings as each main run, no retuning. About 5 to 15 minutes per seed depending on the GPU.
If Colab disconnects, rerun steps 1 to 3, 5 and this cell; finished seeds are skipped.

In [ ]:
for seed in (42, 43, 44):
    cold_dir = f'{OUT}/cold_model0_seed{seed}'
    if os.path.exists(f'{cold_dir}/summary.json'):
        print(f'seed {seed}: already trained')
        continue
    if os.path.exists(cold_dir):
        raise RuntimeError(f'{cold_dir} exists without summary.json (interrupted). Rename it, then rerun this cell.')
    run = RUNS[seed]
    !python -B -m src.evaluation.cold_start train --device cuda \
      --config configs/config.json --grid-index {GRID_INDEX} \
      --reference-run "{run}" --output-dir "{cold_dir}"

## 9. Part F: evaluate the three cold-start copies

In [ ]:
COLD_OUT = f'{OUT}/cold_start_3seeds'
if os.path.exists(COLD_OUT):
    print('Already done:', COLD_OUT)
else:
    !python -B -m src.evaluation.cold_start evaluate --device cuda \
      --config configs/config.json --grid-index {GRID_INDEX} \
      --run-dirs "{OUT}/cold_model0_seed42" "{OUT}/cold_model0_seed43" "{OUT}/cold_model0_seed44" \
      --output-dir "{COLD_OUT}"
display(Markdown(open(f'{COLD_OUT}/cold_start_table.md').read()))

## 10. What was saved
Share the `model0_real_results` folder link with the team.

In [ ]:
!find "{OUT}" -maxdepth 2 | sort